# Attention head — step by step

A minimal, interactive version of `train_attention_head.py`: load the cache, build the `FPGatherHeadModel(AttentionHead)`, run one forward pass, inspect the aggregation, then take training steps one at a time. Run top to bottom; re-run the single-step / loop cells as you like.

**Key knob: `CENTER`.** Absolute window embeddings are dominated by a large shared per-window component; the per-sample signal sits ~4 orders of magnitude below it, and LayerNorm washes it out. Subtracting each window's across-sample mean `m_j` (`window_means`) leaves each window vector *as* the per-sample deviation, which is what makes attention learn. With `CENTER=True` this reaches val_pcc ~0.5 on the absolute hw1000 cache; `False` reproduces the stuck (~0) baseline.

In [ ]:
import sys
from pathlib import Path

import torch

sys.path.insert(0, str(Path.cwd().parents[0]))  # repo root (notebook lives in train_pipeline/)
from crop_embed.models.fp_head_model import AttentionHead, FPGatherHeadModel, window_means
from crop_embed import FixedWindowEmbedder
from crop_embed.data.loading import prepare_data
from crop_embed.train import masked_mse, _compute_metrics

# ── config ──
CACHE       = '../checkpoints/sweep/sativas413_hw1000.ckpt.pt'
SPLIT       = '../splits/sativas413_seed42.pt'
HALF_WINDOW = 1000
BUFFER      = 0
BATCH       = 16
CENTER      = True     # subtract per-window across-sample mean before attention
N_QUERIES   = 8
N_HEADS     = 4
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
device

## 1. Data — dataset, targets, split

In [ ]:
data = prepare_data(split_path=SPLIT, half_window=HALF_WINDOW, buffer=BUFFER)
dataset    = data['dataset']
Y          = data['Y'].to(device)          # (n_samples, n_traits), z-scored, NaN = missing
trait_cols = data['trait_cols']
train_idx  = data['train_idx']
val_idx    = data['val_idx']
print(f'{len(train_idx)} train / {len(val_idx)} val | {Y.shape[1]} traits')

## 2. Cache + per-window mean

`cache` is `(n_fps, D)` (frozen). `sample_fp_index` is `(n_samples, n_windows)`. `window_means(cache, sample_fp_index, train_idx)` returns the `(n_windows, D)` per-window mean across **training** samples — the baseline we subtract so each window vector becomes its per-sample deviation.

In [ ]:
embedder = FixedWindowEmbedder.from_file(CACHE, dataset)
cache           = embedder.cache.float().to(device)        # (n_fps, D), frozen
sample_fp_index = embedder.sample_fp_index.to(device)      # (n_samples, n_windows)
emb_dim   = cache.shape[1]
n_traits  = Y.shape[1]
n_windows = sample_fp_index.shape[1]
assert torch.equal(embedder.sample_fp_index, dataset.sample_fp_index), 'cache/dataset mismatch'

# Per-window mean across TRAIN samples (no val leak). (n_windows, D).
WMEAN = window_means(cache, sample_fp_index, train_idx) if CENTER else None
print(f'cache {tuple(cache.shape)} | {n_windows:,} windows/sample | emb_dim={emb_dim} | n_traits={n_traits}')
if CENTER:
    print(f'window_mean {tuple(WMEAN.shape)}')

## 3. Build the attention head

`FPGatherHeadModel` materializes `cache[gather_ind]` → `(B, n_windows, D)` and feeds it to `AttentionHead`. Passing `window_mean=WMEAN` makes the head subtract `m_j` from each window before attention.

In [ ]:
inner = AttentionHead(emb_dim, n_traits, n_queries=N_QUERIES, n_heads=N_HEADS, window_mean=WMEAN)
head  = FPGatherHeadModel(inner).to(device)
opt   = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
print(f'{sum(p.numel() for p in head.parameters()):,} params | center_windows={inner.center_windows}')

## 4. One forward pass on a batch

In [ ]:
b_idx      = train_idx[:BATCH].to(device)
gather_ind = sample_fp_index[b_idx]          # (B, n_windows)
pred       = head(cache, gather_ind)         # (B, n_traits)
print('pred:', tuple(pred.shape), f'| mean={pred.mean():+.3f} std={pred.std():.3f}')

## 5. Inspect what centering does to the signal

The raw window vector is dominated by the shared per-window component (large magnitude); the per-sample signal is the tiny across-sample std. Centering removes the shared part, so the vector magnitude collapses to ~the signal scale — a big jump in signal-to-background that LayerNorm can then surface.

In [ ]:
with torch.no_grad():
    g = cache[gather_ind]                                # (B, n_windows, D) raw
    print(f'raw      : window |vec|={g.norm(dim=-1).mean():.3f}   std ACROSS samples={g.std(0).mean():.6f}')
    if CENTER:
        c = g - WMEAN
        print(f'centered : window |vec|={c.norm(dim=-1).mean():.3f}   std ACROSS samples={c.std(0).mean():.6f}  (same signal, far less background)')
    # attended summary: does it vary across samples now?
    feats = head.model.attend(g)                         # (B, K*D)
    print(f'attended summary std ACROSS samples: {feats.std(0).mean():.5f}')

## 6. A single training step (re-run this cell repeatedly)

In [ ]:
head.train()
pred = head(cache, gather_ind)
loss = masked_mse(pred, Y[b_idx])
opt.zero_grad(); loss.backward()
gnorm = torch.sqrt(sum(p.grad.pow(2).sum() for p in head.parameters() if p.grad is not None))
opt.step()
print(f'loss={loss.item():.4f}  |grad|={gnorm.item():.3e}')

## 7. Short training loop (with chunked validation)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
EPOCHS = 30
loader = DataLoader(TensorDataset(train_idx), batch_size=BATCH, shuffle=True)

@torch.no_grad()
def val_pcc():
    head.eval()
    preds = [head(cache, sample_fp_index[val_idx[i:i+BATCH].to(device)]) for i in range(0, len(val_idx), BATCH)]
    m = _compute_metrics(torch.cat(preds), Y[val_idx], trait_cols, 'val')
    return m['val/mean_mse'], m['val/mean_pcc']

for ep in range(EPOCHS):
    head.train()
    for (bi,) in loader:
        bi = bi.to(device)
        loss = masked_mse(head(cache, sample_fp_index[bi]), Y[bi])
        opt.zero_grad(); loss.backward(); opt.step()
    vm, vp = val_pcc()
    print(f'epoch {ep:3d}  train_loss={loss.item():.4f}  val_mse={vm:.4f}  val_pcc={vp:+.3f}')